In [2]:
import pandas as pd

RUTA = "../cienciadatos/datos/eod_stgo/"
hogares = pd.read_csv(RUTA + "Hogares.csv", sep=";", decimal=",", low_memory=False)
hogares["Comuna"].value_counts()

Comuna
PUENTE ALTO            1662
MAIPU                  1466
LA FLORIDA             1039
SANTIAGO                998
LAS CONDES              888
SAN BERNARDO            762
PUDAHUEL                630
ÑUÑOA                   603
QUILICURA               570
LA PINTANA              545
PROVIDENCIA             502
PEÑALOLEN               489
EL BOSQUE               415
RECOLETA                385
RENCA                   344
ESTACION CENTRAL        331
CONCHALI                310
CERRO NAVIA             306
LA GRANJA               304
PEDRO AGUIRRE CERDA     296
MACUL                   279
QUINTA NORMAL           278
LA REINA                272
SAN MIGUEL              265
COLINA                  259
VITACURA                254
SAN JOAQUIN             244
MELIPILLA               236
HUECHURABA              235
LO BARNECHEA            234
LA CISTERNA             229
LO PRADO                228
INDEPENDENCIA           225
PEÑAFLOR                225
SAN RAMON               217
CERRILLOS    

Se trabajará con la comuna de Maipú como comuna comuna principal y La Florida como comuna de comparacion

In [3]:
hogares.head()

,Hogar,Sector,Zona,Comuna,DirCoordX,DirCoordY,Fecha,DiaAsig,TipoDia,Temporada,...,NumVeh,NumBicAdulto,NumBicNino,Propiedad,MontoDiv,ImputadoDiv,MontoArr,ImputadoArr,IngresoHogar,Factor
0,100010,7,786,BUIN,335180.8019,6266420.975,14-04-2013,domingo,2,1,...,1,1,0,2,53000.0,0,100000,0,450845,136.393738
1,100020,7,785,BUIN,338410.2114,6265607.141,10-04-2013,miércoles,1,1,...,1,3,0,1,NaN,0,120000,0,1019369,73.843597
2,100030,7,791,BUIN,327863.8248,6257800.086,23-08-2013,viernes,1,1,...,0,0,0,3,NaN,0,70000,0,80000,180.722809
3,100041,7,791,BUIN,327864.0000,6257800.000,23-08-2013,viernes,1,1,...,0,1,0,1,NaN,0,80000,0,559259,150.379059
4,100052,7,783,BUIN,338480.8152,6267296.941,08-08-2013,jueves,1,1,...,0,0,0,1,NaN,0,117771,1,710309,122.001518


#### Hogares en Maipu y la florida

In [5]:
hogaresMaipu = hogares[hogares["Comuna"] == "MAIPU"]
len(hogaresMaipu)

1466

In [6]:
hogaresFlorida = hogares[hogares["Comuna"] == "LA FLORIDA"]
len(hogaresFlorida)

1039

In [7]:
personas = pd.read_csv(RUTA + "personas.csv", sep=";", decimal=",", low_memory=False)
personas["Edad"]= 2026 - personas["AnoNac"]

#### Cantidad de personas en Maipu y La Florida

In [8]:
join_personas_hogares_maipu = pd.merge(hogaresMaipu, personas, on="Hogar", how="left")
len(join_personas_hogares_maipu)

5314

In [9]:
join_personas_hogares_florida= pd.merge(hogaresFlorida, personas, on="Hogar", how="left")
len(join_personas_hogares_florida)

3321

#### Representacion de hogares y personas

In [10]:
join_personas_hogares_maipu["Factor_x"].sum() #representacion de hogares

np.float64(517094.649993)

In [11]:
join_personas_hogares_maipu["Factor_y"].sum() #representacion de personas


np.float64(516375.04553999996)

In [12]:
join_personas_hogares_florida["Factor_x"].sum()

np.float64(383054.12072)

In [13]:
join_personas_hogares_florida["Factor_y"].sum()

np.float64(384580.76622)

En Maipú se encuestaron 1466 hogares que representan aproximadamente 517095 hogares, y 5314 personas que representan aproximadamente 516375 personas

En La Florida se encuestaron a 1039 hogares que representan aproximadamente 383054, a 3321 personas que representan aproximadamente 384581 personas

#### Distribucion de genero (ver)

In [14]:
join_personas_hogares_maipu["Sexo"].map({1: "Hombre", 2: "Mujer"}).value_counts()

Sexo
Mujer     2836
Hombre    2478
Name: count, dtype: int64

In [15]:
join_personas_hogares_florida["Sexo"].map({1: "Hombre", 2: "Mujer"}).value_counts()

Sexo
Mujer     1781
Hombre    1540
Name: count, dtype: int64

In [76]:
join_personas_hogares_maipu[join_personas_hogares_maipu["Sexo"] == 1]["Factor_y"].sum() #representacion de hombres
join_personas_hogares_maipu[join_personas_hogares_maipu["Sexo"] == 2]["Factor_y"].sum() #representacion de mujeres
join_personas_hogares_florida[join_personas_hogares_florida["Sexo"] == 1]["Factor_y"].sum() #representacion de hombres
join_personas_hogares_florida[join_personas_hogares_florida["Sexo"] == 2]["Factor_y"].sum() #representacion de mujeres

np.float64(188318.63954)

In [17]:
def cuantil_ponderado(valores, pesos, cuantiles):
    """Cuantiles de una variable con pesos de muestreo."""
    d = pd.DataFrame({"v": valores, "w": pesos}).dropna().sort_values("v")
    acumulada = d["w"].cumsum() / d["w"].sum()
    return [float(d.loc[(acumulada >= q).idxmax(), "v"]) for q in np.atleast_1d(cuantiles)]

#### Percentiles de edad

##### Media y mediana

In [60]:
import numpy as np

mediaM = np.average(join_personas_hogares_maipu["Edad"])
mediaF = np.average(join_personas_hogares_florida["Edad"])
medianaM = join_personas_hogares_maipu["Edad"].quantile(0.5)
medianaF = join_personas_hogares_florida["Edad"].quantile(0.5)
mediaM, medianaM, mediaF, medianaF

(np.float64(48.9028980052691),
 np.float64(47.0),
 np.float64(51.831376091538694),
 np.float64(51.0))

##### Percentiles 25 y 75

In [61]:
join_personas_hogares_maipu["Edad"].quantile([0.25, 0.75]), join_personas_hogares_florida["Edad"].quantile([0.25, 0.75])

(0.25    31.0
 0.75    65.0
 Name: Edad, dtype: float64,
 0.25    34.0
 0.75    69.0
 Name: Edad, dtype: float64)

##### Percentiles y mediana con Factor

In [62]:
cuantil_ponderado(join_personas_hogares_maipu["Edad"],
                   join_personas_hogares_maipu["Factor_y"], 
                   [0.25, 0.5, 0.75]), cuantil_ponderado(join_personas_hogares_florida["Edad"],
                   join_personas_hogares_florida["Factor_y"],
                     [0.25, 0.5, 0.75])

([31.0, 45.0, 64.0], [32.0, 47.0, 66.0])

#### Media y mediana de los ingresos de los hogares

##### Ponderados

In [23]:
medianaMp =cuantil_ponderado(join_personas_hogares_maipu["IngresoHogar"], join_personas_hogares_maipu["Factor_x"], 0.5)
mediaMp = np.average(join_personas_hogares_maipu["IngresoHogar"], weights=join_personas_hogares_maipu["Factor_x"])
medianaFp = cuantil_ponderado(join_personas_hogares_florida["IngresoHogar"], join_personas_hogares_florida["Factor_x"], 0.5)
mediaFp = np.average(join_personas_hogares_florida["IngresoHogar"], weights=join_personas_hogares_florida["Factor_x"])
medianaMp[0], round(mediaMp), medianaFp[0], round(mediaFp)

(600000.0, 706317, 649210.0, 817662)

##### Sin ponderar

In [29]:
join_personas_hogares_maipu["IngresoHogar"].quantile([0.25, 0.75]), join_personas_hogares_florida["IngresoHogar"].quantile([0.25, 0.75])

(0.25    400000.0
 0.75    860458.0
 Name: IngresoHogar, dtype: float64,
 0.25    375172.0
 0.75    952229.0
 Name: IngresoHogar, dtype: float64)

In [30]:
np.average(join_personas_hogares_maipu["IngresoHogar"]), np.average(join_personas_hogares_florida["IngresoHogar"])

(np.float64(690337.901392548), np.float64(757634.0635350798))

#### Viajes

In [35]:
viajes = pd.read_csv(RUTA + "viajes.csv", sep=";", decimal=",", low_memory=False)
viajes.head()

,Hogar,Persona,Viaje,Etapas,ComunaOrigen,ComunaDestino,SectorOrigen,SectorDestino,ZonaOrigen,ZonaDestino,...,TiempoMedio,Periodo,MinutosDespues,CuadrasDespues,FactorLaboralNormal,FactorSabadoNormal,FactorDomingoNormal,FactorLaboralEstival,FactorFindesemanaEstival,CodigoTiempo
0,173431,17343102,1734310202,1,94.0,94.0,2.0,2.0,400,407,...,3.0,6.0,6.0,1.0,1.000000,NaN,NaN,NaN,NaN,0.0
1,173441,17344101,1734410101,2,94.0,71.0,2.0,3.0,407,307,...,4.0,5.0,5.0,1.0,1.127220,NaN,NaN,NaN,NaN,0.0
2,173441,17344101,1734410102,2,71.0,94.0,3.0,2.0,307,407,...,3.0,5.0,10.0,2.0,1.127220,NaN,NaN,NaN,NaN,0.0
3,173441,17344103,1734410301,2,94.0,91.0,2.0,3.0,407,437,...,2.0,5.0,10.0,2.0,1.127220,NaN,NaN,NaN,NaN,0.0
4,173441,17344103,1734410302,2,91.0,94.0,3.0,2.0,437,407,...,5.0,4.0,10.0,2.0,1.052764,NaN,NaN,NaN,NaN,0.0


In [36]:
viajesMaipu = pd.merge(join_personas_hogares_maipu, viajes, on="Persona", how="left")
viajesMaipu.head()

,Hogar_x,Sector,Zona,Comuna,DirCoordX,DirCoordY,Fecha,DiaAsig,TipoDia,Temporada,...,TiempoMedio,Periodo,MinutosDespues,CuadrasDespues,FactorLaboralNormal,FactorSabadoNormal,FactorDomingoNormal,FactorLaboralEstival,FactorFindesemanaEstival,CodigoTiempo
0,102791,2,418,MAIPU,335966.5733,6284758.905,07-02-2013,jueves,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,102791,2,418,MAIPU,335966.5733,6284758.905,07-02-2013,jueves,1,2,...,3.0,2.0,99.0,0.0,NaN,NaN,NaN,1.143880,NaN,NaN
2,102791,2,418,MAIPU,335966.5733,6284758.905,07-02-2013,jueves,1,2,...,3.0,5.0,0.0,0.0,NaN,NaN,NaN,1.482104,NaN,NaN
3,102801,2,418,MAIPU,336147.2679,6284683.219,08-02-2013,viernes,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,102801,2,418,MAIPU,336147.2679,6284683.219,08-02-2013,viernes,1,2,...,3.0,2.0,0.0,0.0,NaN,NaN,NaN,1.143880,NaN,NaN


In [37]:
viajesFlorida = pd.merge(join_personas_hogares_florida, viajes, on="Persona", how="left")
viajesFlorida.head()

,Hogar_x,Sector,Zona,Comuna,DirCoordX,DirCoordY,Fecha,DiaAsig,TipoDia,Temporada,...,TiempoMedio,Periodo,MinutosDespues,CuadrasDespues,FactorLaboralNormal,FactorSabadoNormal,FactorDomingoNormal,FactorLaboralEstival,FactorFindesemanaEstival,CodigoTiempo
0,118031,6,230,LA FLORIDA,351826.1108,6287711.495,06-10-2012,sábado,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,118031,6,230,LA FLORIDA,351826.1108,6287711.495,06-10-2012,sábado,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,130781,6,233,LA FLORIDA,352362.8344,6285751.910,17-01-2013,jueves,1,2,...,2.0,5.0,5.0,1.0,NaN,NaN,NaN,1.12722,NaN,NaN
3,130781,6,233,LA FLORIDA,352362.8344,6285751.910,17-01-2013,jueves,1,2,...,2.0,5.0,10.0,3.0,NaN,NaN,NaN,1.12722,NaN,NaN
4,130781,6,233,LA FLORIDA,352362.8344,6285751.910,17-01-2013,jueves,1,2,...,2.0,5.0,5.0,1.0,NaN,NaN,NaN,1.12722,NaN,NaN


In [38]:
viajesMaipu["Persona"].value_counts()

Persona
18095301    10
18146201     9
17312001     8
17496202     8
17538202     8
            ..
24406102     1
24406103     1
24406105     1
27341102     1
27341103     1
Name: count, Length: 5314, dtype: int64

In [51]:
factor_totalMaipu = viajesMaipu[["FactorLaboralNormal","FactorFindesemanaEstival","FactorLaboralEstival","FactorDomingoNormal","FactorSabadoNormal"]].fillna(0).sum(axis=1)
factor_totalFlorida = viajesFlorida[["FactorLaboralNormal","FactorFindesemanaEstival","FactorLaboralEstival","FactorDomingoNormal","FactorSabadoNormal"]].fillna(0).sum(axis=1)


##### Viajes sin poderar

In [44]:
viajes_por_persona_Maipu = (
    viajesMaipu.groupby("Persona")["Viaje"].count().reset_index()
)
viajes_por_persona_Maipu

,Persona,Viaje
0,10279101,0
1,10279102,2
2,10280101,0
3,10280102,2
4,10280103,2
...,...,...
5309,24406104,2
5310,24406105,0
5311,27341101,3
5312,27341102,0


In [46]:
viajes_por_persona_Florida = (
    viajesFlorida.groupby("Persona")["Viaje"].count().reset_index()
)
viajes_por_persona_Florida

,Persona,Viaje
0,11803101,0
1,11803102,0
2,13078101,2
3,13078102,2
4,13078103,2
...,...,...
3316,26514102,0
3317,36127101,2
3318,36127102,0
3319,36127103,2


##### Viajes ponderados

In [41]:
viajes_por_persona_Maipu["Viaje"]*=factor_totalMaipu
viajes_por_persona_Maipu

,Persona,Viaje
0,10279101,0.000000
1,10279102,2.287760
2,10280101,0.000000
3,10280102,0.000000
4,10280103,2.287760
...,...,...
5309,24406104,2.105527
5310,24406105,0.000000
5311,27341101,3.381660
5312,27341102,0.000000


In [53]:
viajes_por_persona_Florida["Viaje"]*=factor_totalFlorida
viajes_por_persona_Florida

,Persona,Viaje
0,11803101,0.000000
1,11803102,0.000000
2,13078101,2.254440
3,13078102,2.254440
4,13078103,2.254440
...,...,...
3316,26514102,0.000000
3317,36127101,2.287760
3318,36127102,0.000000
3319,36127103,2.964208


##### Viajes desde Maipu

In [55]:
viajesDesdeMaipu = viajes[viajes["ComunaOrigen"] == 94]
factor_totalMaipu = viajesDesdeMaipu[["FactorLaboralNormal","FactorFindesemanaEstival","FactorLaboralEstival","FactorDomingoNormal","FactorSabadoNormal"]].fillna(0).sum(axis=1)

##### Duracion mediana

In [63]:
viajesDesdeMaipu["TiempoViaje"].quantile(0.5)

np.float64(30.0)

##### Duracion mediana ponderada

In [64]:
cuantil_ponderado(viajesDesdeMaipu["TiempoViaje"], factor_totalMaipu,0.5)[0]

25.0